영화 리뷰 데이터셋을 이용하여 영화제목 예측 분류기를 개선

1. 데이터
- 특성에 품사 정보를 합하여 사용
- 품사는 모든 품사 사용

2. 분류기
- logisticregression
- ridgeclassifier(그리드서치 방법으로 최적의 alpha값 찾기)
- 라쏘 정규화 분류기
- multinomial NB
- n-gram(unigram, bigram, trigram)

3. 다른 매개변수는 성능을 높일 수 있도록 조절
4. 7개 분류기를 실행, 테스트 데이터에 대한 정확도를 비교하여 출력

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from konlpy.tag import Okt
import warnings

warnings.filterwarnings('ignore')

# 1. 데이터 로드 및 타겟/특성 분리
df = pd.read_csv('daum_movie_review.csv')

# 결측치 제거
df = df.dropna(subset=['review', 'title'])

X = df['review']
y = df['title']

# 학습 및 테스트 데이터 분할 (8:2)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 형태소 분석기 초기화
okt = Okt()

# 특성에 품사 정보를 합하여 사용하는 토크나이저 함수 정의 (모든 품사 사용)
def pos_tokenizer(text):
    # 단어와 품사를 "단어/품사" 형태로 결합하여 반환
    return [f"{word}/{pos}" for word, pos in okt.pos(text)]


# ==========================================
# 벡터화 (Vectorization)
# 성능 향상을 위해 min_df, max_df 매개변수를 조절
# ==========================================

# 기본 unigram 벡터라이저 (1~4번 분류기용 및 5번 unigram용)
tfidf_unigram = TfidfVectorizer(tokenizer=pos_tokenizer, ngram_range=(1, 1), min_df=3, max_df=0.9)
X_train_uni = tfidf_unigram.fit_transform(X_train)
X_test_uni = tfidf_unigram.transform(X_test)

# bigram 벡터라이저 (6번 분류기용)
tfidf_bigram = TfidfVectorizer(tokenizer=pos_tokenizer, ngram_range=(1, 2), min_df=3, max_df=0.9)
X_train_bi = tfidf_bigram.fit_transform(X_train)
X_test_bi = tfidf_bigram.transform(X_test)

# trigram 벡터라이저 (7번 분류기용)
tfidf_trigram = TfidfVectorizer(tokenizer=pos_tokenizer, ngram_range=(1, 3), min_df=3, max_df=0.9)
X_train_tri = tfidf_trigram.fit_transform(X_train)
X_test_tri = tfidf_trigram.transform(X_test)


# ==========================================
# 분류기 정의 및 학습 (총 7개)
# ==========================================
results = {}

# 1. Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_uni, y_train)
results['1.LogisticRegression'] = accuracy_score(y_test, lr.predict(X_test_uni))

# 2. Ridge Classifier (그리드서치 방법으로 최적의 alpha값 찾기)
ridge_params = {'alpha': [0.1, 1.0, 10.0, 100.0]}
ridge_grid = GridSearchCV(RidgeClassifier(random_state=42), ridge_params, cv=5, n_jobs=-1)
ridge_grid.fit(X_train_uni, y_train)
best_alpha = ridge_grid.best_params_['alpha']
results[f'2.RidgeClassifier'] = accuracy_score(y_test, ridge_grid.predict(X_test_uni))

# 3. 라쏘 정규화 분류기 (Logistic Regression에 L1 페널티 적용)
lasso = LogisticRegression(penalty='l1', solver='liblinear', max_iter=1000, C=1.0, random_state=42)
lasso.fit(X_train_uni, y_train)
results['3.Lasso'] = accuracy_score(y_test, lasso.predict(X_test_uni))

# 4. Multinomial NB (성능 향상을 위해 alpha=0.1로 조정)
mnb = MultinomialNB(alpha=0.1) 
mnb.fit(X_train_uni, y_train)
results['4.Multinomial NB'] = accuracy_score(y_test, mnb.predict(X_test_uni))

# 5. n-gram (unigram)
# 1번 모델과 동일한 벡터를 사용하므로 로지스틱 회귀를 기준으로 n-gram 성능을 비교
results['5.n-gram (unigram)'] = results['1.LogisticRegression']

# 6. n-gram (bigram)
lr_bi = LogisticRegression(max_iter=1000, random_state=42)
lr_bi.fit(X_train_bi, y_train)
results['6.n-gram (bigram)'] = accuracy_score(y_test, lr_bi.predict(X_test_bi))

# 7. n-gram (trigram)
lr_tri = LogisticRegression(max_iter=1000, random_state=42)
lr_tri.fit(X_train_tri, y_train)
results['7.n-gram (trigram)'] = accuracy_score(y_test, lr_tri.predict(X_test_tri))

print(f"분류기 모델 | {'정확도(Accuracy)'}")
for name, acc in results.items():
    print(f"{name} : {acc:.3f}")

분류기 모델 | 정확도(Accuracy)
1.LogisticRegression : 0.697
2.RidgeClassifier : 0.716
3.Lasso : 0.691
4.Multinomial NB : 0.698
5.n-gram (unigram) : 0.697
6.n-gram (bigram) : 0.691
7.n-gram (trigram) : 0.688
